# 02 – Exploration
Exploratory analysis of DE-LU day-ahead prices and covariate forecasts on the pre-test period (2023-01-01 → 2026-03-31).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config as cfg

df = pd.read_parquet(cfg.DATASET_PATH)
pre = df.loc[:"2026-03-31"]   # pre-test period only

print(pre.shape)
print(pre.index.min(), "→", pre.index.max())
pre.head()


## 1. Series overview and descriptive statistics

In [ ]:
cols = ["price", "load_fc", "wind_fc", "solar_fc"]
labels = {
    "price":    "Day-ahead price\n[EUR/MWh]",
    "load_fc":  "Load forecast\n[GW]",
    "wind_fc":  "Wind forecast\n(on+offshore) [GW]",
    "solar_fc": "Solar forecast\n[GW]",
}

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(14, 10))

for ax, col in zip(axes, cols):
    ax.plot(pre.index, pre[col], lw=0.3, alpha=0.5, label="hourly")
    daily = pre[col].resample("D").mean()
    ax.plot(daily.index, daily, lw=1, color="C1", label="daily mean")
    ax.set_ylabel(labels[col])
    ax.legend(loc="upper right", fontsize=8)

axes[0].axhline(0, color="k", lw=0.5)   # zero line: makes negative prices visible
fig.suptitle("DE-LU day-ahead price and day-ahead forecasts (published on D−1), pre-test period")
fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "series_overview.png", dpi=150)
plt.show()

In [ ]:
pre[cols].describe().round(1)

In [ ]:
p = pre["price"]

print(f"Min price: {p.min():.1f} EUR/MWh on {p.idxmin()}")
print(f"Max price: {p.max():.1f} EUR/MWh on {p.idxmax()}")

**Observations.**
- Most prices are between 67 and 114 EUR/MWh.
- But some prices are extreme: the lowest is −500 EUR/MWh (Sunday 2 July 2023, 14:00: lots of sun, low demand), the highest is 936 EUR/MWh (12 December 2024, 17:00: winter evening, almost no wind).
- Prices are calm in some periods and very unstable in others.
- Load is higher in winter and lower on weekends. Wind changes a lot from day to day. Solar is high in summer.

## 2. Daily and weekly profiles


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [3, 1]})

# Price distribution for each hour of the day
by_hour = [pre.loc[pre["hour"] == h, "price"] for h in range(24)]
axes[0].boxplot(by_hour, showfliers=False)
axes[0].set_xticks(range(1, 25), range(24))
axes[0].set_xlabel("Hour of day (Berlin time)")
axes[0].set_ylabel("Price [EUR/MWh]")
axes[0].set_title("Price by hour of day")

# Price distribution for each day of the week
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_dow = [pre.loc[pre["dow"] == d, "price"] for d in range(7)]
axes[1].boxplot(by_dow, showfliers=False)
axes[1].set_xticks(range(1, 8), days)
axes[1].set_title("Price by day of week")

for ax in axes:
    ax.axhline(0, color="k", lw=0.5)

fig.suptitle("Price profiles, pre-test period (outliers hidden)")
fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "price_profiles_boxplots.png", dpi=150)
plt.show()


In [ ]:
profile = pre.groupby(["non_working_day", "hour"])["price"].mean().unstack(0)
profile.columns = ["Working day", "Non-working day (weekend / holiday)"]

ax = profile.plot(figsize=(10, 4), marker="o", ms=3)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Hour of day (Berlin time)")
ax.set_ylabel("Mean price [EUR/MWh]")
ax.set_title("Mean daily price profile: working vs non-working days")
ax.set_xticks(range(24))
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / "profile_working_vs_nonworking.png", dpi=150)
plt.show()

**Observations.**
- Prices follow the same shape every day: high in the morning (7–8 h), low at midday (13–14 h, because of solar), highest in the evening (19–20 h).
- Prices vary most at midday, and this is also when most negative prices happen.
- Weekends are cheaper, especially Sunday.
- On weekends and holidays, there is no morning peak and midday prices are much lower.
- So the model needs a daily season (s = 24) and the `non_working_day` covariate.

## 3. Price distribution: heavy tails, negative prices and spikes

In [ ]:
import scipy.stats as stats

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(p, bins=200)
axes[0].set_xlabel("Price [EUR/MWh]")
axes[0].set_ylabel("Number of hours")
axes[0].set_title("Histogram of hourly prices")

# QQ plot: if prices were normal, the points would follow the red line
stats.probplot(p, dist="norm", plot=axes[1])
axes[1].set_title("QQ plot against the normal distribution")

fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "price_distribution.png", dpi=150)
plt.show()

In [ ]:
# Spike: price more than 3 standard deviations above the mean
spike_threshold = p.mean() + 3 * p.std()

print(f"Negative-price hours: {(p < 0).sum()} ({(p < 0).mean():.1%} of all hours)")
print(f"Spike hours (> {spike_threshold:.0f} EUR/MWh): {(p > spike_threshold).sum()} ({(p > spike_threshold).mean():.1%} of all hours)")

**Observations.**
- Prices are not normally distributed: the QQ plot moves away from the line at both ends. Extreme prices are much more frequent than for a normal distribution ("heavy tails").
- About 5 % of hours have a negative price.
- Spikes (above ≈ 241 EUR/MWh) are rare: about 0.7 % of hours.
- We keep all these values: they are real.

## 4. Trend / season / noise decomposition

Additive decomposition (prices can be negative), with a period of 168 h (one week), so the season contains both the daily and the weekly cycle.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(p, model="additive", period=168)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

axes[0].plot(p, lw=0.3)
axes[0].set_title("Observed price")

axes[1].plot(decomp.trend, lw=1)
axes[1].set_title("Trend")

# The season repeats every week, so we show only one week (Monday to Sunday)
week = decomp.seasonal.loc["2023-01-02":"2023-01-08"]   # 2023-01-02 is a Monday
axes[2].plot(range(168), week.values)
axes[2].set_xticks(range(0, 168, 24), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[2].set_title("Season (one week shown, it repeats every week)")

axes[3].plot(decomp.resid, lw=0.3)
axes[3].set_title("Noise (residual)")

for ax in axes:
    ax.axhline(0, color="k", lw=0.5)
    ax.set_ylabel("EUR/MWh")

fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "price_decomposition.png", dpi=150)
plt.show()

**Observations.**
- Trend: no clear long-term increase or decrease. The level moves between about 20 and 180 EUR/MWh, with the highest peaks in winter (e.g. December 2024).
- Season: the same weekly pattern repeats: two peaks per weekday (morning and evening), a midday dip, and much lower prices on Saturday and Sunday.
- Noise: large, and it contains all the spikes and negative prices. These are not explained by trend or season.

## 5. Autocorrelation and stationarity

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(14, 4))
plot_acf(p, lags=200, ax=ax)
for lag in [24, 168]:
    ax.axvline(lag, color="C1", ls="--", lw=1)   # expected peaks: 1 day and 1 week
ax.set_xlabel("Lag [hours]")
ax.set_title("ACF of hourly prices (dashed lines: lags 24 and 168)")
fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "price_acf.png", dpi=150)
plt.show()

**Observations.**
- The ACF has a peak every 24 hours: the price at the same hour the day before is similar.
- The peak at lag 168 (one week) is higher than the peaks around it (144, 192): the same hour one week before is also similar.
- The seasonal peaks decrease very slowly: the season is strong and persistent.

**Stationarity** (weak stationarity: stable mean, stable variance, stable dependence):
- Mean: no long-term trend (see the decomposition), so the mean looks roughly stable.
- Variance: not stable. Some periods are calm, others have many spikes.
- Dependence: the strong seasonal peaks in the ACF show a periodic component.
- So the price series is not stationary as it is. The seasonal part must be removed (seasonal differencing) before fitting an ARMA-type model.

## 6. Price vs covariates

In [ ]:
covs = ["load_fc", "wind_fc", "solar_fc"]
corr = pre[covs].corrwith(p)   # Pearson correlation of each covariate with the price

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, col in zip(axes, covs):
    ax.scatter(pre[col], p, s=2, alpha=0.1)
    ax.set_xlabel(f"{col} [GW]")
    ax.set_title(f"Price vs {col} (r = {corr[col]:.2f})")
axes[0].set_ylabel("Price [EUR/MWh]")

fig.tight_layout()
fig.savefig(cfg.FIGURES_DIR / "price_vs_covariates.png", dpi=150)
plt.show()

corr.round(2)

**Observations.**
- Higher load goes with higher prices (r = 0.29).
- More wind (r = −0.37) and more solar (r = −0.43) go with lower prices.
- The very high prices only happen when wind and solar are low.
- The correlations are moderate: no covariate alone explains the price.
- These are correlations only. They do not prove causality: for example, solar and load both follow the time of day, so part of the link can come from the daily cycle.